### We estimate house prices within a zip code based on walk/bike score and commute time to work while controlling for number of bedrooms, (maybe for demographics of the zip too) 
#### Data Sources: 
##### •Zillow Data: House prices at zip level given the number of bedrooms. We use the Fred CPI to inflation adjust the prices. Zillow url sometimes changes. If the code doesn’t run because of this, then the only solution is to manually correct the url.
##### •Walk Score: City level data. Webscraped from https://www.walkscore.com/cities-and-neighborhoods/states/. As the walkscore data is available only for the big cities, only about half of the zip codes were merged.

##### •Census: Demographics and commute time at zip code level. As census data uses zcta instead of zip, there is a very small proportion of non-merged data. 

Regression:
Regress 2023-08-31 on bedroom, walk score, bike score, 'pct_black', 'pct_less_HS', 'pct_bsc_higher', 'Long commute' and also maybe some info on past prices in the same zip?



In [ ]:
#Import Packages
import requests
import pandas as pd
from io import StringIO
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import io


In [ ]:
# To get multiple outputs at once
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
#Set Path
#import os

# Zillow

In [ ]:
# Zillow Data: Data is downloaded from here:https://www.zillow.com/research/data/ However, url to download the file changes often. If the url doesn't exist, update the path before running the
# Create an empty list to store dataframes
house_prices_list = []

#download the data from Zillow website
for number in range(1, 6):
    url_zillow = f'https://files.zillowstatic.com/research/public_csvs/zhvi/Zip_zhvi_bdrmcnt_{number}_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv?t=1696158077'
    response = requests.get(url_zillow)
    data = pd.read_csv(StringIO(response.text))
    #add a 'number of bedrooms' column 
    data['Bedroom'] = number
    
    # Append the dataframe to the list
    house_prices_list.append(data)

# Concatenate the dataframes in the list
house_prices = pd.concat(house_prices_list, ignore_index=True)
house_prices.head()

In [ ]:
#Rename RegionName to zip
house_prices=house_prices.rename(columns={'RegionName': 'zip'})

In [ ]:
#Adjust house prices by yearly inflation (https://fred.stlouisfed.org/series/CPIAUCSL, Index 1982-1984=100, seasonally adj)
## Import CPI from fed
url_CPI = "https://fred.stlouisfed.org/graph/fredgraph.csv?bgcolor=%23e1e9f0&chart_type=line&drp=0&fo=open%20sans&graph_bgcolor=%23ffffff&height=450&mode=fred&recession_bars=on&txtcolor=%23444444&ts=12&tts=12&width=1318&nt=0&thu=0&trc=0&show_legend=yes&show_axis_titles=yes&show_tooltip=yes&id=CPIAUCSL&scale=left&cosd=1947-01-01&coed=2023-08-01&line_color=%234572a7&link_values=false&line_style=solid&mark_type=none&mw=3&lw=2&ost=-99999&oet=99999&mma=0&fml=a&fq=Monthly&fam=avg&fgst=lin&fgsnd=2020-02-01&line_index=1&transformation=lin&vintage_date=2023-09-26&revision_date=2023-09-26&nd=1947-01-01"
CPI = pd.read_csv(url_CPI)
#Prep the CPI data to be used to adjust house prices
CPI['DATE'] = CPI['DATE'].str[:-3] 
#CPI['DATE'] = CPI['DATE'].astype(str) + '_CPI'
CPI = CPI.transpose()
# Set the 'date' variable to be the column names
CPI.columns = CPI.iloc[0]
CPI = CPI.iloc[1:]
CPI.head()

In [ ]:
#Create CPI adjusted house prices data
for col in house_prices.columns:
    #Extract the first 7 characters of house_prices data frame
    first7chr = col[:7]
    #Find the columns from CPI that has the same first 7 ch
    if first7chr in CPI.columns:
        #Divide the corresponding house prices column to CPI and multip by 100 to adjust for inflation
        #house_prices_CPI=house_prices.copy()
        house_prices[col] = (house_prices[col] / CPI[first7chr].iloc[0]) * 100 


In [ ]:
#Sanity check
house_prices.head()
house_prices.tail()

In [ ]:
#Understand the missing data
house_prices['2023-08-31'].isna().sum() #no missing in 2023 august
house_prices['2023-07-31'].isna().sum()
house_prices['2023-06-30'].isna().sum()


In [ ]:
#Descriptive stats of house_prices
pd.set_option('display.float_format', '{:,.2f}'.format)
house_prices_descriptive=house_prices[['Bedroom','2023-06-30','2023-07-31','2023-08-31']].describe()
house_prices_descriptive

In [ ]:
#Make a time series plot of the house prices in the zip with most houses for sale: i.e. sizeRank=1
print(house_prices.columns) #SizeRank column gives the zip with the most houses on sale
biggest_zip_rows=house_prices[house_prices['SizeRank'] == 1] #filter the rows where SizeRank=1
biggest_zip = biggest_zip_rows['zip'].iloc[0]
house_prices.Bedroom[house_prices['zip'] == biggest_zip] #there are no 1 bedroom houses for sale in this area

## Locate the state and the city of the zip with most houses for sale:
State_biggest_zip=house_prices.State[house_prices['zip'] == biggest_zip].iloc[0]
State_biggest_zip

City_biggest_zip=house_prices.City[house_prices['zip'] == biggest_zip].iloc[0]
City_biggest_zip

##Create the Plot
import datetime
bedroom_counts = range(2, 6)

##Create subplots for bedrooms from 2 to 6 (there are no 1 bedroom houses listed in this zip)
fig, axes = plt.subplots(len(bedroom_counts), 1, figsize=(8, 6 * len(bedroom_counts)), gridspec_kw={'hspace': 0.5})

#Create a dataframe with 2-bedroom house prices in the biggest zip 
biggest_zip_2bedroom= house_prices[(house_prices['zip'] == biggest_zip) & (house_prices['Bedroom'] == 2)].iloc[:, 9:-1]

for i in bedroom_counts:
 #Select the df given bedroom count
    house_prices_values = house_prices[(house_prices['zip'] == biggest_zip) & (house_prices['Bedroom'] == i)].iloc[:, 9:-1]
    house_prices_values = house_prices_values.iloc[0].values
    
    dates = pd.to_datetime(biggest_zip_2bedroom.columns)
    current_date = datetime.datetime.now().strftime("%B %Y")

    axes[i - 2].plot(dates, house_prices_values, linestyle='-')  # Subtract 2 to match the index
    axes[i - 2].set_title(f'{i}-Bedroom House Prices in the Most Listed Zip Code')
    axes[i - 2].set_xlabel('Date')
    axes[i - 2].set_ylabel('House Prices (CPI adjusted, 1982-1984=100)')
    
    axes[i - 2].annotate(f"Note: Most listed zip code as of {current_date} is {biggest_zip} in the state {State_biggest_zip}, city {City_biggest_zip}", xy=(0.5, -0.2), xycoords="axes fraction", fontsize=9, ha="center")


In [ ]:
#Bar Plot of the cities that contain the zip codes with most expensive 2-bedroom houses
two_bedroom_houses = house_prices[house_prices['Bedroom'] == 2]
sorted_top_10_prices = two_bedroom_houses.sort_values(by='2023-08-31', ascending=False).head(10)

# Select the top 10 cities and their corresponding house prices
top_10_city_zip = 'zip ' + sorted_top_10_prices['zip'].astype(str) + ' in ' + sorted_top_10_prices['City']
top_10_prices = sorted_top_10_prices['2023-08-31']

# Create a bar plot
plt.figure
plt.ticklabel_format(axis='y', style='plain') 
plt.bar(top_10_city_zip, top_10_prices)
plt.title('Zip Codes with the Most Expensive 2-Bedroom Houses')
plt.xlabel('Zip, City')
plt.ylabel('House Prices (CPI adjusted prices 1982=100)')
plt.xticks(rotation=90)

plt.show()


In [ ]:
#Bar Plot of the cities that contain the zip codes with cheapest 2-bedroom houses
two_bedroom_houses = house_prices[house_prices['Bedroom'] == 2]
sorted_bottom_10_prices = two_bedroom_houses.sort_values(by='2023-08-31', ascending=True).head(10)

# Select the top 10 cities and their corresponding house prices
bottom_10_city_zip = 'zip ' + sorted_bottom_10_prices['zip'].astype(str) + ' in ' + sorted_bottom_10_prices['City']
bottom_10_prices = sorted_bottom_10_prices['2023-08-31']

# Create a bar plot
plt.figure
plt.ticklabel_format(axis='y', style='plain') 
plt.bar(bottom_10_city_zip, bottom_10_prices)
plt.title('Zip Codes with the Leasy Expensive 2-Bedroom Houses')
plt.xlabel('Zip, City')
plt.ylabel('House Prices (CPI adjusted prices 1982=100)')
plt.xticks(rotation=90)

plt.show()

In [ ]:
#Create variables based on past house prices (potentially to be used for regression)

##Create yearly average house prices variables
for year in range(2021, 2024):
    year_columns = [col for col in house_prices.columns if col.startswith(str(year))]
    house_prices[f'Average_{year}'] = house_prices[year_columns].mean(axis=1)
    
## Growth of house prices
for year in range(2022, 2024):
    previous_year = year - 1
    
    # Calculate the percentage growth from the previous year to the current year
    house_prices[f'Growth_{previous_year}_to_{year}'] = (
        house_prices[f'Average_{year}'] / house_prices[f'Average_{previous_year}'] - 1
    )


In [ ]:
# Organize the data

#Drop the columns we don't need
## Drop columns from 2000 to 2018
pattern = r'^(2000|2001|2002|2003|2004|2005|2006|2007|2008|2009|2010|2011|2012|2013|2014|2015|2016|2017|2018|2019|2020|2021|2022)'

columns_to_drop = house_prices.filter(regex=pattern)
house_prices.drop(columns=columns_to_drop.columns, inplace=True)

#drop other unnecessary columns
columns_to_drop = ['RegionID', 'RegionType', 'StateName', 'Metro']

# Drop the specified columns
house_prices.drop(columns=columns_to_drop, inplace=True)

In [ ]:
#Sanity Check:
house_prices.head()

# Walk Index

In [ ]:
#Walk/Transit/Bike Score: We scrape the data from walkscore.com

# First we check that website for all the states have only one table per webpage. 
state= [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "DC", "FL", "GA", "HI", "ID", "IL", "IN", "IA", "KS", "KY",
    "LA", "ME", "MD", "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ", "NM", "NY", "NC", "ND", 
    "OH", "OK", "OR", "PA", "RI", "SC", "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY"
]

for st in state:
    url = f"https://www.walkscore.com/{st}"
    df_list = pd.read_html(url)
    num_tables = len(df_list)  # Number of tables on the website
    
    if num_tables == 1:
        print(f"Webpage for the state {st} has only 1 table.")
    else:
        print("More than one table")


In [ ]:
#Now we can scrape the tables
state= [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA", "HI", "ID", "IL", "IN", "IA", "KS", "KY",
    "LA", "ME", "MD", "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ", "NM", "NY", "NC", "ND", 
    "OH", "OK", "OR", "PA", "RI", "SC", "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY"
]

#Initialize an empty data frame
walk_score = []

for st in state:
    url = f"https://www.walkscore.com/{st}"
    tables= pd.read_html(url)
    walk_score += [pd.DataFrame(table) for table in tables]
    for table in tables:
        table['State'] = st

walk_score = pd.concat(walk_score)


In [ ]:
#Sanity Check
walk_score.head()
type(walk_score) 

In [ ]:
#Clean the data
##Get rid of the extra info with () at the end of some cities 
walk_score['City'] = walk_score['City'].str.replace(r'\s*\([^)]*\)', '', regex=True)

## Drop unnecessary variables:
walk_score = walk_score.drop(columns=['Population', 'Zip Code'])


In [ ]:
#Sanity Check:
walk_score.head()

In [ ]:
# Bar Plot of the Most Walkable Cities
walk_score_top10 = walk_score.sort_values(by=['Walk Score', 'Bike Score', 'Transit Score'], ascending=False).head(10)
width = 0.35
x = np.arange(len(walk_score_top10))

# Plot Walk Score and Bike Score bars 
plt.bar(x - width/2, walk_score_top10['Walk Score'], width=bar_width, label='Walk Score')
plt.bar(x + width/2, walk_score_top10['Bike Score'], width=bar_width, label='Bike Score')
#plt.bar(x + width/2, walk_score_bottom10['Transit Score'], width=bar_width, label='Transit Score', alpha=1)

plt.xlabel('City')
plt.ylabel('Scores')
plt.title('Most Walkable Cities in the US')
plt.xticks(indices, walk_score_top10['City'], rotation=90)
plt.legend()

plt.show()

In [ ]:
#Bar plot of the Least Walkable Cities
walk_score_bottom10 = walk_score.sort_values(by=['Walk Score', 'Bike Score', 'Transit Score'], ascending=True).head(10)
width = 0.35
x = np.arange(len(walk_score_bottom10))

# Plot Walk Score and Bike Score bars 
plt.bar(x - width/2, walk_score_bottom10['Walk Score'], width=bar_width, label='Walk Score')
plt.bar(x + width/2, walk_score_bottom10['Bike Score'], width=bar_width, label='Bike Score')
#plt.bar(x + width/2, walk_score_bottom10['Transit Score'], width=bar_width, label='Transit Score', alpha=1)

plt.xlabel('City')
plt.ylabel('Scores')
plt.title('Least Walkable Cities in the US')
plt.xticks(indices, walk_score_bottom10['City'], rotation=90)
plt.legend()

plt.show()

In [ ]:
#Scatter Plot of Walk vs Bike Score
plt.scatter(walk_score['Walk Score'], walk_score['Bike Score'], alpha=0.5) 
plt.ylabel('Bike Score')
plt.xlabel('Walk Score')
plt.title('Scatter Plot of Bike Score vs. Walk Score')


In [ ]:
#Merge Walk Index to Zillow
house_prices = pd.merge(house_prices, walk_score, on=['City', 'State'], how='left')

In [ ]:
#Descriptive stats on the merge
total_obs=house_prices.shape[0]
non_merged=house_prices['Walk Score'].isnull().sum()
print(f"Total number of observations in house_prices: {total_obs}")
print(f"Number of non-merged observations: {non_merged}") #As the walk score index were computed for big cities only,small cities were not merged. 

#We can impute the non-merged cities with the state level average? or we can leave it as it is and run the regression only for the zip codes in big cities. 

In [ ]:
#Scatter Plot of House Prices and Walk Score
plt.scatter(house_prices['Walk Score'], house_prices['2023-08-31'], alpha=0.5)
plt.title('Scatter Plot of Walk Score vs. House Prices')
plt.xlabel('Walk Score')
plt.ylabel('House Prices at the Date 2023-08-31')

plt.show()


In [ ]:
#Scatter Plot of House Prices and Bike Score
plt.scatter(house_prices['Bike Score'], house_prices['2023-08-31'], alpha=0.5)
plt.title('Scatter Plot of Bike Score vs. House Prices')
plt.xlabel('Bike Score')
plt.ylabel('House Prices at the Date 2023-08-31')

# Show the plot
plt.show()

# Census Data: Demographic Controls

In [ ]:
import zipfile

url_census_demog = "https://data.census.gov/api/access/table/download?download_id=0e22908ebd288452ea74e7450c2625aba74cace52e17cd1431542d638249e4f9"
zip_file = "ACSST5Y2021.S2502_2023-10-02T031911.zip"

response = requests.get(url_census_demog)

with open(zip_file, "wb") as file:
    file.write(response.content)
csv_file_in_zip = "ACSST5Y2021.S2502-Data.csv"

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    with zip_ref.open(csv_file_in_zip) as file_in_zip:
        csv_data = file_in_zip.read()
demographics = pd.read_csv(io.BytesIO(csv_data))

demographics.head()


In [ ]:
#Organize the data
demographics = demographics.drop(0).reset_index(drop=True)
demographics = demographics.rename(columns={'NAME': 'zip'})
#S2502_C02_011E= pct of home owners younger than 35
#S2502_C02_003E= pct of black or african american owned houses
#S2502_C02_018E=pct of house owners that are less than high school grad
#S2502_C02_021E=pct of house owners that have BSc or higher
demographics = demographics[["S2502_C02_011E", "S2502_C02_003E", "S2502_C02_018E", "S2502_C02_021E", "zip"]]
demographics['zip'] = demographics['zip'].str[6:]
demographics.head()

In [ ]:
#Rename var:
#S2502_C02_011E= pct of home owners younger than 35
#S2502_C02_003E= pct of black or african american owned houses
#S2502_C02_018E=pct of house owners that are less than high school grad
#S2502_C02_021E=pct of house owners that have BSc or higher

demographics = demographics.rename(columns={
    'S2502_C02_011E': 'pct_younger_35',
    'S2502_C02_003E': 'pct_black',
    'S2502_C02_018E': 'pct_less_HS',
    'S2502_C02_021E': 'pct_bsc_higher'})

In [ ]:
#Join the demographics data into house prices data
house_prices['zip'] = house_prices['zip'].astype(str)
census['zip'] = demographics['zip'].astype(str)

house_prices= pd.merge(house_prices, demographics, on='zip', how='left')

In [ ]:
house_prices.head()

In [ ]:
#Check how successful the merge was
missing_demog=house_prices['pct_younger_35'].isna().sum()
print("Number of unmatched observations", missing_demog) #match is not perfect bc Zip and Zcta area codes sometimes don't match.

In [ ]:
import statsmodels
import pandas

print("statsmodels version:", statsmodels.__version__)
print("pandas version:", pandas.__version__)


# Census: Commute Data

In [ ]:
import zipfile

url_census_commute = "https://data.census.gov/api/access/table/download?download_id=278c69b8e17a5e52ae49ed6323c631452c762747b6ea30c1d39e6ab4e4681fe6"
zip_file = "ACSDT5Y2021.B08303_2023-10-02T063015.zip"

response = requests.get(url_census_commute)


with open(zip_file, "wb") as file:
    file.write(response.content)
csv_file_in_zip = "ACSDT5Y2021.B08303-Data.csv"

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    with zip_ref.open(csv_file_in_zip) as file_in_zip:
        csv_data = file_in_zip.read()
commute = pd.read_csv(io.BytesIO(csv_data))

#Organize the data
commute = commute.drop(0).reset_index(drop=True)
commute = commute.rename(columns={'NAME': 'zip'})
commute['zip'] = commute['zip'].str[6:]

commute.head()


In [ ]:
# Rename the columns to remove special characters
commute.rename(columns={
    'B08303_003E': 'Less than 5 min',
    'B08303_004E': '5 to 9 min',
    'B08303_005E': '10 to 14 min',
    'B08303_006E': '15 to 19 min',
    'B08303_007E': '20 to 24 min',
    'B08303_008E': '25 to 29 min',
    'B08303_009E': '30 to 34 min',
    'B08303_010E': '35 to 39 min',
    'B08303_011E': '40 to 44 min',
    'B08303_012E': '45 to 59 min',
    'B08303_013E': '60 to 89 min'
}, inplace=True)

relevant_columns=['Less than 5 min', '5 to 9 min', '10 to 14 min', '15 to 19 min',
                    '20 to 24 min', '25 to 29 min', '30 to 34 min', '35 to 39 min',
                    '40 to 44 min', '45 to 59 min', '60 to 89 min']
commute=commute[['zip'] + relevant_columns]

commute['Total'] = commute[relevant_columns].sum(axis=1)
commute['Total'].replace(0, np.nan, inplace=True)

for col in relevant_columns:
    commute[col] = pd.to_numeric(commute[col], errors='coerce')
    
commute['Total'].replace(0, np.nan, inplace=True)
commute['Total'] = pd.to_numeric(commute['Total'], errors='coerce')
commute['Short Commute'] = (commute['Less than 5 min'] + commute['5 to 9 min'] + commute['10 to 14 min']) / commute['Total'] * 100
commute['Moderate Commute'] = (commute['15 to 19 min'] + commute['20 to 24 min'] + commute['25 to 29 min'] + commute['30 to 34 min'] + commute['35 to 39 min']) / commute['Total'] * 100
commute['Long Commute'] = (commute['45 to 59 min'] + commute['60 to 89 min'] + commute['40 to 44 min']) / commute['Total'] * 100

commute=commute[['Short Commute', 'Moderate Commute', 'Long Commute', 'zip']]

In [ ]:
commute.head()

In [ ]:
house_prices= pd.merge(house_prices, commute, on='zip', how='left')

In [ ]:
plt.scatter(house_prices['2023-08-31'], house_prices['Long Commute'], alpha=0.5)


In [ ]:
house_prices